In [1]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth
from mlxtend.frequent_patterns import association_rules
import pandas as pd
import ast

In [2]:
df_carts = pd.read_csv("../../data/tesco/tesco_carts_clean.csv")
df_inventory = pd.read_csv("../../data/tesco/tesco_inventory_clean.csv")

In [3]:
df_carts.head()

,cart_id,cart
0,0,[73314923]
1,1,"[58175124, 50502269, 70943424, 57346955, 56036..."
2,2,"[50962501, 50503297, 50507984, 50169663, 51965..."
3,3,"[72680530, 62284801, 58098273]"
4,4,"[56373768, 67336474, 52844621, 54921285]"


In [4]:
df_inventory.head()

,product_id,category,description,ingredients,energy,fat,saturates,salt,sugars,protein,carbohydrate,fibre,avg_price
0,68238698,fruit_veg,Tesco Baby Corn 190G (M),no_ingredients,28.0,0.4,0.1,0.3,1.9,2.5,2.7,2.0,1.645
1,53426251,fruit_veg,Tesco Cranberries 100G,Pineapple Juice from Concentrate Cranberries S...,336.0,1.6,0.2,0.1,65.0,0.3,77.4,5.5,1.495
2,59445495,fruit_veg,Tesco Crispy Slices 350G,Potato (78%) Batter Rapeseed Oil Batter contai...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.500
3,74533881,sweets,Kelloggs Pop Tarts Frosted S'mores 416G,"Enriched Flour (Wheat Flour, Niacin, Reduced I...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.500
4,54198489,grains,Osem Bamba Peanut Snack 25G,Peanuts (50%) Corn Palm Oil Salt,534.0,34.0,6.6,1.0,3.2,0.0,0.0,0.0,0.400


To learn more about their customers’ behavior, the Tesco marketing department is interested in three
pieces of information.
- A list of triplets of products (A, B, C) such that C is purchased frequently after A and B. They are
interested in strong associations, such that the likelihood of the association is at least 1.8 times higher
than chance.
- A list of pairs (A, B) such that B is frequently purchased after A, and that the pair (A, B) is found
least in 0.5% of the carts.
- The top three items by confidence in the association rule A → B,

In [5]:
# Inspecting the carts dataset carts that contain more than a single item is a list but the list is in quotation marks which turns it into a string instead of a list
# which I can verify by simply grabbing one cart which multiple items and making python determine its type.
print(type(df_carts['cart'][1]))

<class 'str'>


In [6]:
# To convert the string representation of a list to a list I use ast.literal_eval, inspired from here: https://stackoverflow.com/questions/1894269/how-to-convert-string-representation-of-list-to-a-list
# and from what we did in exercises for lecture 5.
carts = df_carts['cart'].apply(ast.literal_eval).tolist()

In [7]:
# Verify that the same cart as before now actually has type list
print(type(carts[1]))

<class 'list'>


In [8]:
print(len(carts))

1582801


In [9]:
te = TransactionEncoder()
te_data = te.fit(carts).transform(carts, sparse=True)
df = pd.DataFrame.sparse.from_spmatrix(te_data, columns=te.columns_)

# product indices must either start from 0 or be strings
df.columns = [str(i) for i in df.columns] 

In [10]:
# Very low minimum support because the dataset has 1.5m entries, a value of 0.05 found no triplets
frequent_itemsets = apriori(df, min_support=0.005, use_colnames=True, low_memory=True)

In [11]:
frequent_itemsets

,support,itemsets
0,0.006962,frozenset({50068523})
1,0.007201,frozenset({50264481})
2,0.007234,frozenset({50314549})
3,0.005459,frozenset({50342538})
4,0.005873,frozenset({50461944})
...,...,...
154,0.007180,"frozenset({50502436, 77454543})"
155,0.005193,"frozenset({50503441, 50550228})"
156,0.005349,"frozenset({54550994, 50503441})"
157,0.005358,"frozenset({50503441, 62816594})"


In [12]:
rules = association_rules(frequent_itemsets, metric='lift', min_threshold=0.001)
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({50502436}),frozenset({50502269}),0.054489,0.088677,0.016375,0.300528,3.389015,1.0,0.011543,1.302872,0.745553,0.129153,0.232465,0.242596
1,frozenset({50502269}),frozenset({50502436}),0.088677,0.054489,0.016375,0.184664,3.389015,1.0,0.011543,1.159658,0.773523,0.129153,0.137676,0.242596
2,frozenset({50503441}),frozenset({50502269}),0.036397,0.088677,0.009737,0.267528,3.016878,1.0,0.006510,1.244174,0.693783,0.084424,0.196254,0.188666
3,frozenset({50502269}),frozenset({50503441}),0.088677,0.036397,0.009737,0.109805,3.016878,1.0,0.006510,1.082463,0.733584,0.084424,0.076181,0.188666
4,frozenset({50550228}),frozenset({50502269}),0.020380,0.088677,0.006243,0.306352,3.454697,1.0,0.004436,1.313812,0.725321,0.060725,0.238856,0.188379
5,frozenset({50502269}),frozenset({50550228}),0.088677,0.020380,0.006243,0.070406,3.454697,1.0,0.004436,1.053815,0.779679,0.060725,0.051067,0.188379
6,frozenset({50652534}),frozenset({50502269}),0.024788,0.088677,0.007079,0.285569,3.220325,1.0,0.004880,1.275592,0.706997,0.066537,0.216050,0.182697
7,frozenset({50502269}),frozenset({50652534}),0.088677,0.024788,0.007079,0.079824,3.220325,1.0,0.004880,1.059811,0.756562,0.066537,0.056436,0.182697
8,frozenset({50689433}),frozenset({50502269}),0.032198,0.088677,0.007034,0.218472,2.463686,1.0,0.004179,1.166079,0.613869,0.061791,0.142425,0.148899
9,frozenset({50502269}),frozenset({50689433}),0.088677,0.032198,0.007034,0.079326,2.463686,1.0,0.004179,1.051188,0.651914,0.061791,0.048696,0.148899


In [17]:
# Code inspired by https://rasbt.github.io/mlxtend/user_guide/frequent_patterns/association_rules/
rules["antecedent_len"] = rules["antecedents"].apply(lambda x: len(x))
rules["consequents_len"] = rules["consequents"].apply(lambda x: len(x))
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedent_len,consequents_len
0,frozenset({50502436}),frozenset({50502269}),0.054489,0.088677,0.016375,0.300528,3.389015,1.0,0.011543,1.302872,0.745553,0.129153,0.232465,0.242596,1,1
1,frozenset({50502269}),frozenset({50502436}),0.088677,0.054489,0.016375,0.184664,3.389015,1.0,0.011543,1.159658,0.773523,0.129153,0.137676,0.242596,1,1
2,frozenset({50503441}),frozenset({50502269}),0.036397,0.088677,0.009737,0.267528,3.016878,1.0,0.006510,1.244174,0.693783,0.084424,0.196254,0.188666,1,1
3,frozenset({50502269}),frozenset({50503441}),0.088677,0.036397,0.009737,0.109805,3.016878,1.0,0.006510,1.082463,0.733584,0.084424,0.076181,0.188666,1,1
4,frozenset({50550228}),frozenset({50502269}),0.020380,0.088677,0.006243,0.306352,3.454697,1.0,0.004436,1.313812,0.725321,0.060725,0.238856,0.188379,1,1
5,frozenset({50502269}),frozenset({50550228}),0.088677,0.020380,0.006243,0.070406,3.454697,1.0,0.004436,1.053815,0.779679,0.060725,0.051067,0.188379,1,1
6,frozenset({50652534}),frozenset({50502269}),0.024788,0.088677,0.007079,0.285569,3.220325,1.0,0.004880,1.275592,0.706997,0.066537,0.216050,0.182697,1,1
7,frozenset({50502269}),frozenset({50652534}),0.088677,0.024788,0.007079,0.079824,3.220325,1.0,0.004880,1.059811,0.756562,0.066537,0.056436,0.182697,1,1
8,frozenset({50689433}),frozenset({50502269}),0.032198,0.088677,0.007034,0.218472,2.463686,1.0,0.004179,1.166079,0.613869,0.061791,0.142425,0.148899,1,1
9,frozenset({50502269}),frozenset({50689433}),0.088677,0.032198,0.007034,0.079326,2.463686,1.0,0.004179,1.051188,0.651914,0.061791,0.048696,0.148899,1,1


In [20]:
rules[ (rules['antecedent_len'] >= 2) &
       (rules['lift'] > 1.8) ]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedent_len,consequents_len


In [21]:
rules[ (rules['antecedent_len'] == 1) & 
       (rules['consequents_len'] == 1) &
       (rules['support'] >= 0.005) ]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedent_len,consequents_len
0,frozenset({50502436}),frozenset({50502269}),0.054489,0.088677,0.016375,0.300528,3.389015,1.0,0.011543,1.302872,0.745553,0.129153,0.232465,0.242596,1,1
1,frozenset({50502269}),frozenset({50502436}),0.088677,0.054489,0.016375,0.184664,3.389015,1.0,0.011543,1.159658,0.773523,0.129153,0.137676,0.242596,1,1
2,frozenset({50503441}),frozenset({50502269}),0.036397,0.088677,0.009737,0.267528,3.016878,1.0,0.006510,1.244174,0.693783,0.084424,0.196254,0.188666,1,1
3,frozenset({50502269}),frozenset({50503441}),0.088677,0.036397,0.009737,0.109805,3.016878,1.0,0.006510,1.082463,0.733584,0.084424,0.076181,0.188666,1,1
4,frozenset({50550228}),frozenset({50502269}),0.020380,0.088677,0.006243,0.306352,3.454697,1.0,0.004436,1.313812,0.725321,0.060725,0.238856,0.188379,1,1
5,frozenset({50502269}),frozenset({50550228}),0.088677,0.020380,0.006243,0.070406,3.454697,1.0,0.004436,1.053815,0.779679,0.060725,0.051067,0.188379,1,1
6,frozenset({50652534}),frozenset({50502269}),0.024788,0.088677,0.007079,0.285569,3.220325,1.0,0.004880,1.275592,0.706997,0.066537,0.216050,0.182697,1,1
7,frozenset({50502269}),frozenset({50652534}),0.088677,0.024788,0.007079,0.079824,3.220325,1.0,0.004880,1.059811,0.756562,0.066537,0.056436,0.182697,1,1
8,frozenset({50689433}),frozenset({50502269}),0.032198,0.088677,0.007034,0.218472,2.463686,1.0,0.004179,1.166079,0.613869,0.061791,0.142425,0.148899,1,1
9,frozenset({50502269}),frozenset({50689433}),0.088677,0.032198,0.007034,0.079326,2.463686,1.0,0.004179,1.051188,0.651914,0.061791,0.048696,0.148899,1,1


In [23]:
# 53482131 = id of Kettle Chips Sea Salt And Black Pepper Corns 150G
# probably need to first conver the item ids into names

# This atleast does not seem to be working, but could also just be because I have not actually set the min support low enough originally?
rules[ (rules['antecedents'] == frozenset({53482131})) &
       (rules['lift'] >= 1.05) ]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedent_len,consequents_len
